# Diff-MoE on BabyLM -- Kaggle T4 x2 training notebook

Runs one config from `docs/plan.md`'s 2x2 ablation (standard/differential attention x dense/MoE FFN) on Kaggle's dual-T4 notebook, using both GPUs via DDP. Training data is the [BabyLM Challenge](https://babylm.github.io/) corpus: six domains (child-directed speech, adult conversation, literary prose, subtitles, Wikipedia, telephone dialogue), released as one text file per domain rather than a single blob -- which is what makes it a meaningful testbed for whether MoE experts specialize.

**Before running**: set Notebook Settings -> Accelerator = **GPU T4 x2**.

**Session limits**: 12h max, ~30 GPU-h/week (same quota whether you pick T4 x1 or x2). Checkpoints save every `ckpt_freq` steps to `/kaggle/working/checkpoints` -- copy that folder to a Kaggle Dataset before your session ends so the next session can resume.

**Workflow per run**: (1) clone repo, (2) prepare tokenized data once and re-use across sessions via a Kaggle Dataset, (3) throughput probe on both GPUs, (4) train with DDP, (5) inspect metrics + report.

## 1. Setup

In [1]:
!git clone -b rebuild https://github.com/ramprasathk07/Differential-MOE.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 207, done.
remote: Counting objects: 100% (207/207), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 207 (delta 82), reused 181 (delta 59), pack-reused 0 (from 0)
Receiving objects: 100% (207/207), 374.03 KiB | 3.82 MiB/s, done.
Resolving deltas: 100% (82/82), done.
/kaggle/working/repo


In [2]:
# Staleness guard: Kaggle keeps its OWN copy of this notebook, independent of the
# repo. If either side is behind, cells pass flags or name configs the other side
# doesn't have -- which surfaces later as a confusing FileNotFoundError on
# train.bin rather than as "your notebook is out of date". Check both directions
# up front, while it is still cheap to fix.
import glob
import subprocess

_help = subprocess.run(['python', '-m', 'src.data.train_tokenizer', '--help'],
                       capture_output=True, text=True).stdout
_missing = []
if '--track' not in _help:
    _missing.append('src.data.train_tokenizer has no --track (pre-BabyLM code)')
if '--pretrained' not in _help:
    _missing.append('src.data.train_tokenizer has no --pretrained (pre-tier-S code)')
if not glob.glob('configs/s_*.yaml'):
    _missing.append('configs/s_*.yaml absent (tier S configs missing)')

if _missing:
    raise SystemExit(
        'STALE CODE/NOTEBOOK MISMATCH:\n  - ' + '\n  - '.join(_missing) + '\n\n'
        'The clone is behind this notebook. Fix whichever applies:\n'
        '  * the clone cell points at a branch without these changes -- check -b <branch>\n'
        '  * /kaggle/working/repo is a stale checkout from an earlier session --\n'
        '    `rm -rf /kaggle/working/repo` and re-run the clone cell\n'
        '  * or this notebook is newer than the pushed code -- push first'
    )
print('OK: cloned code matches this notebook (--track, --pretrained, tier-S configs all present).')
print('tier S configs found:', sorted(glob.glob('configs/s_*.yaml')))

OK: cloned code matches this notebook (--track, --pretrained, tier-S configs all present).
tier S configs found: ['configs/s_dense.yaml', 'configs/s_diff.yaml', 'configs/s_diffmoe.yaml', 'configs/s_moe.yaml']


In [3]:
import torch
n_gpu = torch.cuda.device_count()
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', n_gpu)
for i in range(n_gpu):
    print(f'  cuda:{i}', torch.cuda.get_device_name(i))
if n_gpu < 2:
    print('\nWARNING: fewer than 2 GPUs visible -- set Notebook Settings > Accelerator = GPU T4 x2, '
          'then Session > Restart & Run All. The --ddp cells below need nproc_per_node=2 to match n_gpu.')

CUDA available: True
GPU count: 2
  cuda:0 Tesla T4
  cuda:1 Tesla T4


In [4]:
from kaggle_secrets import UserSecretsClient
import wandb

# Get API key from Kaggle Secrets
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb")  # Replace with your secret name if different

# Login
wandb.login(key=wandb_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ramkan103 (New_103) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 2. Run settings

Change these per run -- no code edits needed elsewhere in the notebook.

In [5]:
import glob
import os

# --- pick ONE config -------------------------------------------------------
# tier A  (~16M, custom 4k BPE):  a_dense / a_diff / a_moe / a_diffmoe
# tier B  (~55M, custom 8k BPE):  b_final
# tier S (~295M, cl100k frontier): s_dense / s_diff / s_moe / s_diffmoe
CONFIG = 'configs/s_diffmoe.yaml'

WANDB_PROJECT = 'diff-moe-kaggle'  # change freely per experiment batch
USE_WANDB = True
N_GPU = 2  # matches Accelerator = GPU T4 x2; set to 1 for the single-T4 option

_name = CONFIG.split('/')[-1].replace('.yaml', '')
TIER = _name[0]  # 'a', 'b' or 's'

# Tier S uses a pretrained frontier tokenizer (no BPE training step) and the
# 100M-word 'strict' track; tiers A/B train a custom BPE on 'strict-small'.
if TIER == 's':
    TRACK, _fallback_dir = 'strict', '/kaggle/working/data_s'
    TOKENIZER = 'hf:Xenova/gpt-4'   # cl100k. For Qwen: 'hf:Qwen/Qwen2.5-0.5B'
                                     #   -> then set vocab_size: 151680 in the s_*.yaml configs
else:
    TRACK = 'strict' if TIER == 'b' else 'strict-small'
    _fallback_dir = '/kaggle/working/data_b' if TIER == 'b' else '/kaggle/working/data'
    TOKENIZER = None  # trained below by the sweep + train step

# Prefer tokenized data attached as a Kaggle Dataset. Tokenizing is CPU-only, so
# doing it in a GPU session burns quota on work a laptop does just as well -- and
# /kaggle/working is wiped between sessions, so it would be repeated every time.
# See docs/kaggle-dataset-upload.md for preparing it locally and uploading once.
def _find_attached_data():
    """An attached input dir holding train.bin + meta.json, at ANY depth.
    Recursive: Kaggle mounts a dataset under /kaggle/input/<slug>/, but the
    upload's own folder is kept below that, and owner/slug layouts nest several
    levels deep (e.g. /kaggle/input/datasets/<owner>/<slug>/data_s/). The old
    shallow glob only looked one and two levels down, so it missed those and
    printed a misleading 'not found'."""
    for meta in sorted(glob.glob('/kaggle/input/**/meta.json', recursive=True)):
        d = os.path.dirname(meta)
        if os.path.exists(os.path.join(d, 'train.bin')):
            return d
    return None

_attached = _find_attached_data()
DATA_DIR = _attached or _fallback_dir      # set this by hand to override
USING_ATTACHED = _attached is not None

# wandb auth: without this, rank0's wandb.init() blocks at a login prompt inside
# the non-interactive training cell. Add-ons > Secrets > WANDB_API_KEY.
if USE_WANDB:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
        print('wandb key loaded from Kaggle secret WANDB_API_KEY')
    except Exception as e:
        print('WARNING: no WANDB_API_KEY Kaggle secret -- the training cell will hang at wandb login.')
        print('Fix: Add-ons > Secrets > add WANDB_API_KEY, or set USE_WANDB = False. Details:', e)

run_name = _name
wandb_flag = f'--wandb --wandb_project {WANDB_PROJECT}' if USE_WANDB else ''
ddp_prefix = f'torchrun --standalone --nproc_per_node={N_GPU}' if N_GPU > 1 else 'python'
ddp_flag = '--ddp' if N_GPU > 1 else ''

print(f'\nrun_name: {run_name} | tier: {TIER} | track: {TRACK}')
print(f'tokenizer: {TOKENIZER or "custom BPE (trained below)"}')
print(f'data_dir : {DATA_DIR}')
if USING_ATTACHED:
    print('           ^ attached Kaggle Dataset -- tokenization will be SKIPPED (no GPU time spent)')
else:
    print('           ^ no attached dataset found; the next cells will tokenize here.')
    print('             For tier S that is ~13 min of CPU inside a GPU session. Prefer preparing')
    print('             locally and attaching it -- see docs/kaggle-dataset-upload.md.')
print('launch prefix:', ddp_prefix)

wandb key loaded from Kaggle secret WANDB_API_KEY

run_name: s_diffmoe | tier: s | track: strict
tokenizer: hf:Xenova/gpt-4
data_dir : /kaggle/input/datasets/ramprasathk07/babylm-challenge/data_s
           ^ attached Kaggle Dataset -- tokenization will be SKIPPED (no GPU time spent)
launch prefix: torchrun --standalone --nproc_per_node=2


In [6]:
# Kaggle input resolver: supports extracted Dataset files and an uploaded data_s.zip.
# Set DATASET_PATH to an exact directory or .zip path only when auto-detection
# is ambiguous. Kaggle inputs are read-only; zip contents are copied to working.
import json
import shutil
import zipfile
from pathlib import Path

DATASET_PATH = None  # e.g. '/kaggle/input/babylm-tokenized/data_s.zip'
REQUIRED_DATA_FILES = ('train.bin', 'val.bin', 'test.bin', 'meta.json',
                       'domain_offsets.json')

def _compatible_data_dir(path):
    path = Path(path)
    if not all((path / name).is_file() for name in REQUIRED_DATA_FILES):
        return False
    try:
        meta = json.loads((path / 'meta.json').read_text())
    except (OSError, ValueError):
        return False
    if meta.get('track') != TRACK:
        return False
    if TOKENIZER is not None and meta.get('tokenizer_spec') != TOKENIZER:
        return False
    return True

def _extract_token_data(zip_path):
    destination = Path('/kaggle/working/attached_data')
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        members = {Path(name).name: name for name in archive.namelist()}
        missing = [name for name in REQUIRED_DATA_FILES if name not in members]
        if missing:
            raise RuntimeError(f'{zip_path} is missing required files: {missing}')
        for name in REQUIRED_DATA_FILES:
            with archive.open(members[name]) as src, open(destination / name, 'wb') as dst:
                shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
    return destination

def _resolve_kaggle_data(explicit=None):
    if explicit:
        selected = Path(explicit)
        if not selected.exists():
            raise FileNotFoundError(f'DATASET_PATH does not exist: {selected}')
        candidates = [_extract_token_data(selected)] if selected.is_file() else [selected]
    else:
        input_root = Path('/kaggle/input')
        candidates = sorted({p.parent for p in input_root.rglob('meta.json')}) \
            if input_root.exists() else []
        candidates = [p for p in candidates if _compatible_data_dir(p)]
        if not candidates and input_root.exists():
            archives = sorted(input_root.rglob('*.zip'))
            for archive in archives:
                try:
                    extracted = _extract_token_data(archive)
                except (OSError, zipfile.BadZipFile, RuntimeError):
                    continue
                if _compatible_data_dir(extracted):
                    candidates = [extracted]
                    break
    compatible = [p for p in candidates if _compatible_data_dir(p)]
    if len(compatible) > 1:
        raise RuntimeError('Multiple compatible token datasets found; set DATASET_PATH: ' +
                           ', '.join(map(str, compatible)))
    return compatible[0] if compatible else None

resolved_data = _resolve_kaggle_data(DATASET_PATH)
if resolved_data is not None:
    DATA_DIR = str(resolved_data)
    USING_ATTACHED = True
else:
    DATA_DIR = _fallback_dir
    USING_ATTACHED = False

# Never start two torchrun workers when Kaggle only exposed one GPU.
N_GPU = max(1, n_gpu)
if n_gpu == 0:
    print('WARNING: no GPU is visible; training will run on CPU.')
elif n_gpu < 2:
    print('WARNING: only one GPU is visible; using single-process training.')

# A missing secret must not leave wandb waiting for an interactive login.
if USE_WANDB and not os.environ.get('WANDB_API_KEY'):
    USE_WANDB = False
    print('W&B disabled because WANDB_API_KEY is unavailable.')

wandb_flag = f'--wandb --wandb_project {WANDB_PROJECT}' if USE_WANDB else ''
ddp_prefix = f'torchrun --standalone --nproc_per_node={N_GPU}' if N_GPU > 1 else 'python'
ddp_flag = '--ddp' if N_GPU > 1 else ''
print(f'Using data: {DATA_DIR}')
print(f'Launch: {ddp_prefix} {ddp_flag}')


Using data: /kaggle/input/datasets/ramprasathk07/babylm-challenge/data_s
Launch: torchrun --standalone --nproc_per_node=2 --ddp


## 3. Tokenizer + data

**Preferred: attach pre-tokenized data as a Kaggle Dataset.** Tokenizing is CPU-only, so running it inside a GPU session spends quota on work a laptop does just as well — and `/kaggle/working` is wiped between sessions, so it would be repeated every single time. Prepare it once locally and upload it (`docs/kaggle-dataset-upload.md`); the settings cell finds any attached input containing `train.bin` + `meta.json` automatically and skips straight past this section.

For the `strict` track that saves ~13 minutes of GPU-session time per run, and it is the same data every run reads, so preparing it once is also what keeps the four ablation runs strictly comparable.

**Fallback**: with nothing attached, these cells tokenize into `/kaggle/working`. Tier A/B trains a custom BPE first (the sweep prints the fertility table to pick a vocab size from); tier S skips training entirely because a pretrained frontier tokenizer already has its vocabulary.

Either way the last cell runs `src.data.verify`, which is cheap and catches the failures that are otherwise invisible until a training run has already burned hours: a wrong dtype, a truncated upload, or a config whose `vocab_size` cannot cover the data.

In [7]:
import os

# Tier A/B: adaptive vocab sweep (docs/plan.md SS2) -- read the fertility table,
# pick a vocab_size, set it below. BabyLM's six domains are heterogeneous, so
# don't reuse a vocab size chosen for a different corpus.
# Tier S: skipped -- it uses a pretrained frontier tokenizer, nothing to train.
# The --pretrained flag scores them side by side, which is what justifies the
# choice (lower fertility, but far more embedding params -- see docs/plan.md).
if TOKENIZER is not None:
    print(f'tier S: using pretrained {TOKENIZER}, no BPE training needed.')
    print('(optional) compare it against custom vocabs on this corpus:')
    print(f'  !python -m src.data.train_tokenizer --sweep --candidates 4096 16384 '
          f'--pretrained {TOKENIZER} --track {TRACK} --max_lines_per_domain 20000')
elif not os.path.exists(f'{DATA_DIR}/train.bin'):
    !python -m src.data.train_tokenizer --sweep --candidates 2048 4096 8192 16384 --track {TRACK} --max_lines_per_domain 20000
else:
    print('data already prepared at', DATA_DIR)

tier S: using pretrained hf:Xenova/gpt-4, no BPE training needed.
(optional) compare it against custom vocabs on this corpus:
  !python -m src.data.train_tokenizer --sweep --candidates 4096 16384 --pretrained hf:Xenova/gpt-4 --track strict --max_lines_per_domain 20000


In [8]:
import json, yaml

# VOCAB_SIZE applies to tiers A/B ONLY -- it is the size of the custom BPE this
# cell trains. Tier S ignores it completely: a pretrained tokenizer already has
# its vocabulary (cl100k = 100,263), which is why the s_*.yaml configs declare
# vocab_size: 100352 (that value padded up to a multiple of 128).
VOCAB_SIZE = 4096  # tier A/B only; take it from the sweep table above

cfg_vocab = yaml.safe_load(open(CONFIG))['model']['vocab_size']
if TOKENIZER is not None:
    print(f'tier {TIER}: pretrained {TOKENIZER} -- VOCAB_SIZE above is unused.')
    print(f'          {CONFIG} declares vocab_size: {cfg_vocab}')
else:
    print(f'tier {TIER}: training a custom BPE at vocab_size {VOCAB_SIZE}')
    print(f'          {CONFIG} declares vocab_size: {cfg_vocab}')
    if VOCAB_SIZE != cfg_vocab:
        print(f'  WARNING: mismatch -- the config wants {cfg_vocab}. train.py will refuse '
              f'to start if the config cannot cover the data.')

if USING_ATTACHED:
    print(f'\nusing attached dataset at {DATA_DIR} -- nothing to tokenize.')
elif not os.path.exists(f'{DATA_DIR}/train.bin'):
    # Tokenizing the full 'strict' track writes ~800MB of uint32 .bin files and
    # takes ~13 min of CPU. Doing it here spends GPU-session time on CPU work and
    # is lost when the session ends -- prefer preparing locally and attaching it.
    if TOKENIZER is not None:                      # tier S: pretrained, no training step
        !python -m src.data.prepare --tokenizer {TOKENIZER} --out_dir {DATA_DIR} --track {TRACK}
    else:                                          # tier A/B: train the custom BPE first
        !python -m src.data.train_tokenizer --vocab_size {VOCAB_SIZE} --out {DATA_DIR}/tokenizer.json --track {TRACK}
        !python -m src.data.prepare --tokenizer {DATA_DIR}/tokenizer.json --out_dir {DATA_DIR} --track {TRACK}
else:
    print('\ndata already prepared at', DATA_DIR)

# Verify before spending GPU hours on it. Tokenized data fails silently -- a wrong
# dtype or a truncated upload decodes as noise rather than raising, and only shows
# up as a loss that will not come down. This also confirms CONFIG's vocab_size can
# cover the data, which train.py would otherwise refuse to start on.
print()
!python -m src.data.verify --data_dir {DATA_DIR} --config {CONFIG}

tier s: pretrained hf:Xenova/gpt-4 -- VOCAB_SIZE above is unused.
          configs/s_diffmoe.yaml declares vocab_size: 100352

using attached dataset at /kaggle/input/datasets/ramprasathk07/babylm-challenge/data_s -- nothing to tokenize.

tokenizer : hf:Xenova/gpt-4
vocab     : 100263 (max id seen 100257)
dtype     : uint32
track     : strict

tokenizer_config.json: 100%|███████████████████| 460/460 [00:00<00:00, 1.19MB/s]
vocab.json: 2.01MB [00:00, 25.8MB/s]
merges.txt: 917kB [00:00, 68.5MB/s]
special_tokens_map.json: 100%|████████████████| 98.0/98.0 [00:00<00:00, 374kB/s]
tokenizer.json: 4.23MB [00:00, 81.9MB/s]
train:  167,534,939 tokens    639.1MB  dtype=uint32  max_id=100257
       sample: "Well it's just that, you know, a pound, or a hundred pounds today, is not the same as a hundred poun"
  val:   17,287,735 tokens     65.9MB  dtype=uint32  max_id=100257
       sample: "Doctor  's obviously a more frugal character than me because he had this room and the radiator was t"
 test: 

## 4. Throughput probe (docs/plan.md Phase 3)

Runs ~100 steps across both GPUs and reports tok/s, so the token budget and wall-clock are **measured, not guessed**. `batch_size` in the config is **per-GPU**, so with `N_GPU=2` the effective global batch doubles automatically (see `effective_global_batch_tokens` in `report.json`).

For tier S this probe is doing real work beyond timing — it is the first thing that would OOM. At vocab 100k the fp32 logits tensor is `batch x seq x vocab x 4B`, which is why `s_*.yaml` ships a small `batch_size: 4`. If the probe survives with memory to spare, you can raise `batch_size` and halve `accum_steps` to cut step overhead; if it OOMs, halve `batch_size` and double `accum_steps` (global batch stays constant either way).

The cell after the probe converts measured throughput into a concrete `max_steps` for your quota — **use that number**, not the 2900 placeholder in the config, which came from an unvalidated 25 TFLOP/s estimate.

In [9]:
import yaml, os
# ---- smaller tier-S geometry -------------------------------------------------
# Cut dim + depth (the resident-param + activation drivers). vocab & seq_len kept
# so eval NLL stays comparable to any 512-ctx baseline. Tweak DIM / N_LAYERS and
# re-run to go smaller.
DIM        = 768     # was 896
N_LAYERS   = 14      # was 16
N_HEADS    = 12      # head_dim = DIM/N_HEADS must be EVEN (differential attn halves it)
N_EXPERTS  = 6
TOP_K      = 2
BATCH_SIZE = 2
ACCUM      = 32      # global batch = BATCH_SIZE*ACCUM*N_GPU, unchanged from 4*32*2

head_dim = DIM // N_HEADS
assert DIM % N_HEADS == 0 and head_dim % 2 == 0, f'head_dim {head_dim} must be even'
INTER        = 4 * DIM          # dense FFN width
EXPERT_INTER = INTER // TOP_K   # top_k * expert_inter = inter -> MoE active == dense

geom = dict(dim=DIM, n_layers=N_LAYERS, n_heads=N_HEADS, inter_dim=INTER)
moe  = dict(n_experts=N_EXPERTS, top_k=TOP_K, expert_inter_dim=EXPERT_INTER)

CFG_DIR = '/kaggle/working/repo/configs'
for name in ['s_dense', 's_diff', 's_moe', 's_diffmoe']:   # ALL 4 -> stays matched
  path = f'{CFG_DIR}/{name}.yaml'
  if not os.path.exists(path):
      print('skip (not found):', path); continue
  cfg = yaml.safe_load(open(path))
  cfg['model'].update(geom)
  if cfg['model'].get('ffn') == 'moe':
      cfg['model'].update(moe)
  cfg['train'].update(batch_size=BATCH_SIZE, accum_steps=ACCUM)
  yaml.safe_dump(cfg, open(path, 'w'), sort_keys=False)
  print(f'{name:10s} dim={DIM} layers={N_LAYERS} heads={N_HEADS} '
        f'seq_len={cfg["model"]["seq_len"]}(kept) ffn={cfg["model"]["ffn"]} '
        f'batch={BATCH_SIZE} accum={ACCUM}')

# exact param counts for the new geometry (never eyeball these)
print()
os.system('cd /kaggle/working/repo && python -m src.params '
        '--config configs/s_dense.yaml configs/s_moe.yaml configs/s_diffmoe.yaml')

s_dense    dim=768 layers=14 heads=12 seq_len=512(kept) ffn=dense batch=2 accum=64
s_diff     dim=768 layers=14 heads=12 seq_len=512(kept) ffn=dense batch=2 accum=64
s_moe      dim=768 layers=14 heads=12 seq_len=512(kept) ffn=moe batch=2 accum=64
s_diffmoe  dim=768 layers=14 heads=12 seq_len=512(kept) ffn=moe batch=2 accum=64

config                      attn          ffn            raw    active   non-emb raw  non-emb active
----------------------------------------------------------------------------------------------------
configs/s_dense.yaml        standard      dense      209.21M   209.21M       132.14M         132.14M
configs/s_moe.yaml          standard      moe        379.14M   209.27M       302.07M         132.20M
configs/s_diffmoe.yaml      differential  moe        379.16M   209.29M       302.09M         132.22M


0

In [10]:
# # NOTE: probe writes to its own out_dir so its checkpoints / LR history / 'best'
# # selections don't pollute the real run (which would otherwise auto-resume from
# # the probe's step-100 checkpoint with a mismatched cosine schedule).
# #
# # expandable_segments:True reduces allocator fragmentation. Tier-S MoE fills the
# # T4 almost to the brim (all experts + fp32 AdamW state), so fragmentation alone
# # can trip an OOM even when the totals fit -- this reclaims the reserved-but-
# # unallocated slack. Harmless for the dense/diff runs.
# !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True {ddp_prefix} -m src.train --config {CONFIG} --data_dir {DATA_DIR} --out_dir /kaggle/working/probe {ddp_flag} --max_steps 100

In [11]:
# Turn the probe's measured tok/s into a max_steps you can actually afford.
# Reads the probe's own metrics.csv rather than asking you to eyeball the log.
import pandas as pd, yaml, json, os

HOURS_PER_RUN = 7.5   # 30h weekly quota / 4 ablation runs
N_RUNS = 4

probe_csv = f'/kaggle/working/probe/{run_name}/metrics.csv'
cfg = yaml.safe_load(open(CONFIG))
tok_per_step = (cfg['train']['batch_size'] * cfg['model']['seq_len']
                * cfg['train']['accum_steps'] * N_GPU)

if os.path.exists(probe_csv):
    df = pd.read_csv(probe_csv)
    rate = df['tok_per_sec'].dropna()
    # drop the first logged point: it carries warmup/compile cost
    measured = rate.iloc[1:].median() if len(rate) > 1 else rate.iloc[0]
    affordable = int(measured * HOURS_PER_RUN * 3600 / tok_per_step)

    meta = json.load(open(f'{DATA_DIR}/meta.json')) if os.path.exists(f'{DATA_DIR}/meta.json') else {}
    train_tokens = meta.get('tokens', {}).get('train')

    print(f'measured        : {measured:,.0f} tok/s  ({tok_per_step:,} tok/step)')
    print(f'config max_steps: {cfg["train"]["max_steps"]}  '
          f'-> {cfg["train"]["max_steps"]*tok_per_step/3600/measured:.1f} h')
    print(f'affordable      : {affordable} steps in {HOURS_PER_RUN}h '
          f'({affordable*tok_per_step/1e6:,.0f}M tokens)')
    if train_tokens:
        print(f'                  = {affordable*tok_per_step/train_tokens:.1f} epochs of this corpus '
              f'({train_tokens/1e6:.0f}M tokens)')
        print('  >4 epochs starts to repeat data heavily; consider fewer steps or more data.')
    print(f'\nquota check     : {N_RUNS} runs x {HOURS_PER_RUN}h = {N_RUNS*HOURS_PER_RUN}h')
    print(f'\n-> set max_steps: {affordable} in {CONFIG} before the full run')
else:
    print('run the probe cell first')

run the probe cell first


## 5. Full training run

Re-running this cell auto-resumes from `checkpoints/<run_name>/last.pt` if it exists -- safe to re-run after a Kaggle session restart. Rank-0-only logging/checkpointing/wandb is handled inside `src/train.py`; nothing extra needed here.

In [12]:
# expandable_segments:True: same fragmentation guard as the probe cell -- tier-S
# MoE runs sit right at the T4 memory ceiling, so keep it on for the full run too.
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True {ddp_prefix} -m src.train --config {CONFIG} --data_dir {DATA_DIR} --out_dir /kaggle/working/checkpoints {ddp_flag} {wandb_flag}

W0722 16:31:24.384000 157 torch/distributed/run.py:852] 
W0722 16:31:24.384000 157 torch/distributed/run.py:852] *****************************************
W0722 16:31:24.384000 157 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0722 16:31:24.384000 157 torch/distributed/run.py:852] *****************************************
[W722 16:31:24.686715141 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W722 16:31:27.235556144 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W722 16:31:27.235874921 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[s_diffmoe] raw=379.16M active=209.29M world_size=2
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Curre

## 6. Inspect metrics + report

In [13]:
import json
with open(f'/kaggle/working/checkpoints/{run_name}/report.json') as f:
    print(json.dumps(json.load(f), indent=2))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/checkpoints/s_diffmoe/report.json'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f'/kaggle/working/checkpoints/{run_name}/metrics.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.dropna(subset=['train_loss']).plot(x='step', y='train_loss', ax=axes[0], title='train loss')
df.dropna(subset=['val_nll']).plot(x='step', y='val_nll', ax=axes[1], title='val NLL', marker='o')
plt.tight_layout()
plt.show()
df.tail(10)

## 7. Params table (for README / blog tables -- never hand-compute)

In [ ]:
# Never hand-compute these for a README/blog table -- regenerate them.
# Confirms the two parity claims the ablation rests on: standard vs differential
# attention cost the same, and dense vs MoE cost the same *active* params.
print('--- tier A (~16M, custom 4k BPE) ---')
!python -m src.params --config configs/a_dense.yaml configs/a_diff.yaml configs/a_moe.yaml configs/a_diffmoe.yaml
print('\n--- tier S (~295M active, cl100k) ---')
!python -m src.params --config configs/s_dense.yaml configs/s_diff.yaml configs/s_moe.yaml configs/s_diffmoe.yaml

## 8. Persist checkpoints across sessions

Kaggle wipes `/kaggle/working` between sessions. 'Save Version' preserves everything under `/kaggle/working` as this notebook's output. To resume in a NEXT session: attach that output (or a Dataset made from it) as an input, then **copy it back into `/kaggle/working`** before training -- `/kaggle/input` is read-only, so pointing `--out_dir` at it directly would crash on the first checkpoint save.

In [ ]:
# This session's checkpoints (saved automatically when you 'Save Version'):
!ls -la /kaggle/working/checkpoints/{run_name}/ 2>/dev/null || echo 'no checkpoints yet'
print("best/ holds only the top-2 checkpoints (max_best_checkpoints in config); last.pt is always kept for resume.")

# NEXT session, to resume: attach the previous version's output as an input dataset,
# then copy it into the writable working dir BEFORE running the training cell
# (/kaggle/input is READ-ONLY -- training must never write there). Uncomment + edit:
# !mkdir -p /kaggle/working/checkpoints
# !cp -r /kaggle/input/<your-ckpt-dataset>/checkpoints/* /kaggle/working/checkpoints/